In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col, regexp_replace, trim, lower

# 1. Criar o Banco de Dados / Schema para a Camada Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS protege_bronze COMMENT 'Camada Bronze - Dados Brutos da PNAUM (Fase 2)'")

print("🚀 Iniciar Ingestão de Microdados PNAUM (Camada Bronze)...")

# 2. Carga dos dados com mock realista da PNAUM (APS)
# Trazendo dados de idosos (>=60) e adultos para teste do filtro populacional
data_pnaum = [
    (1001, 68, "Feminino", "Hipertensao; Insuficiencia Renal", "Ibuprofeno", "AINE", 30, "UBS Centro"),
    (1002, 75, "Masculino", "Demencia; Risco Quedas", "Clonazepam", "Benzodiazepinico", 60, "UBS Bairro"),
    (1003, 62, "Feminino", "Diabetes Mellitus", "Metformina", "Antidiabetico", 15, "UBS Norte"),
    (1004, 81, "Masculino", "Artrite Reumatoide; Ulcera", "Aspirina", "AINE", 45, "UBS Sul"),
    (1005, 55, "Feminino", "Hipertensao", "Enalapril", "Anti-hipertensivo", 10, "UBS Centro"), # Adulto (<60)
    (1006, 70, "Feminino", "Doenca Renal Cronica", "Diclofenaco", "AINE", 20, "UBS Leste"),
    (1007, 83, "Masculino", "Alzheimer; Historico Quedas", "Diazepam", "Benzodiazepinico", 90, "UBS Centro")
]

columns = ["id_paciente", "idade", "sexo", "morbidades_relatadas", "medicamento_nome", "classe_terapeutica", "dias_uso", "unidade_saude"]

df_raw = spark.createDataFrame(data_pnaum, columns)

print(f"📊 Registros na carga inicial: {df_raw.count()}")

# 3. Sanitização Estrutural e Filtro Populacional (Idade >= 60 anos)
df_bronze = df_raw.filter(col("idade") >= 60) \
    .withColumn("medicamento_sanitizado", trim(lower(regexp_replace(col("medicamento_nome"), "[^a-zA-Z0-9 ]", "")))) \
    .withColumn("morbidades_sanitizadas", trim(lower(col("morbidades_relatadas")))) \
    .withColumn("_ingestao_timestamp", current_timestamp()) \
    .withColumn("_origem_fonte", col("unidade_saude"))

print(f"🎯 Registros mantidos para a população geriátrica (>= 60 anos): {df_bronze.count()}")

# 4. Salvar na Tabela Delta no Schema protege_bronze
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("protege_bronze.pnaum_atendimentos_bruto")

print("✅ Tabela Delta protege_bronze.pnaum_atendimentos_bruto salva com sucesso!")

# 5. Exibir os dados salvos
display(spark.table("protege_bronze.pnaum_atendimentos_bruto"))

🚀 Iniciar Ingestão de Microdados PNAUM (Camada Bronze)...
📊 Registros na carga inicial: 7
🎯 Registros mantidos para a população geriátrica (>= 60 anos): 6
✅ Tabela Delta protege_bronze.pnaum_atendimentos_bruto salva com sucesso!


id_paciente,idade,sexo,morbidades_relatadas,medicamento_nome,classe_terapeutica,dias_uso,unidade_saude,medicamento_sanitizado,morbidades_sanitizadas,_ingestao_timestamp,_origem_fonte
1001,68,Feminino,Hipertensao; Insuficiencia Renal,Ibuprofeno,AINE,30,UBS Centro,ibuprofeno,hipertensao; insuficiencia renal,2026-10-14T11:56:19.295Z,UBS Centro
1002,75,Masculino,Demencia; Risco Quedas,Clonazepam,Benzodiazepinico,60,UBS Bairro,clonazepam,demencia; risco quedas,2026-10-14T11:56:19.295Z,UBS Bairro
1003,62,Feminino,Diabetes Mellitus,Metformina,Antidiabetico,15,UBS Norte,metformina,diabetes mellitus,2026-10-14T11:56:19.295Z,UBS Norte
1004,81,Masculino,Artrite Reumatoide; Ulcera,Aspirina,AINE,45,UBS Sul,aspirina,artrite reumatoide; ulcera,2026-10-14T11:56:19.295Z,UBS Sul
1006,70,Feminino,Doenca Renal Cronica,Diclofenaco,AINE,20,UBS Leste,diclofenaco,doenca renal cronica,2026-10-14T11:56:19.295Z,UBS Leste
1007,83,Masculino,Alzheimer; Historico Quedas,Diazepam,Benzodiazepinico,90,UBS Centro,diazepam,alzheimer; historico quedas,2026-10-14T11:56:19.295Z,UBS Centro


In [0]:
# Célula para popular 10.000 registros da PNAUM na Camada Bronze
import random
from pyspark.sql.functions import current_timestamp, col, regexp_replace, trim, lower

print("🔄 Gerando e ingerindo volume populacional da PNAUM (10.000 registros)...")

# Listas de amostragem epidemiológica da PNAUM
morbidades_list = [
    "Hipertensao; Insuficiencia Renal Crônica", 
    "Demencia; Risco Elevado de Quedas", 
    "Diabetes Mellitus Tipo 2", 
    "Artrite Reumatoide; Ulcera Peptica", 
    "Insuficiencia Renal", 
    "Alzheimer; Historico de Quedas e Tontura",
    "Hipertensao Arterial",
    "Sem comorbidades graves registradas"
]

meds_list = [
    ("Ibuprofeno", "AINE"), ("Clonazepam", "Benzodiazepinico"), 
    ("Metformina", "Antidiabetico"), ("Aspirina", "AINE"), 
    ("Diclofenaco", "AINE"), ("Diazepam", "Benzodiazepinico"),
    ("Enalapril", "Anti-hipertensivo"), ("Omeprazol", "Inibidor Bomba"),
    ("Losartana", "Anti-hipertensivo"), ("Alprazolam", "Benzodiazepinico")
]

ubs_list = ["UBS Centro", "UBS Bairro Norte", "UBS Sul", "UBS Leste", "UBS Vila Real"]

# Gerando 10.000 registros aleatórios e realistas
random.seed(42)
data_pnaum_grande = []

for i in range(1, 10001):
    id_pac = 10000 + i
    idade = random.randint(55, 95) # Inclui adultos e idosos
    sexo = random.choice(["Feminino", "Masculino"])
    morb = random.choice(morbidades_list)
    med, classe = random.choice(meds_list)
    dias = random.choice([15, 30, 60, 90, 180])
    ubs = random.choice(ubs_list)
    
    data_pnaum_grande.append((id_pac, idade, sexo, morb, med, classe, dias, ubs))

columns = ["id_paciente", "idade", "sexo", "morbidades_relatadas", "medicamento_nome", "classe_terapeutica", "dias_uso", "unidade_saude"]

df_raw_large = spark.createDataFrame(data_pnaum_grande, columns)

# Aplicação da regra do P.R.O.T.E.G.E. (Filtro Idade >= 60)
df_bronze_large = df_raw_large.filter(col("idade") >= 60) \
    .withColumn("medicamento_sanitizado", trim(lower(regexp_replace(col("medicamento_nome"), "[^a-zA-Z0-9 ]", "")))) \
    .withColumn("morbidades_sanitizadas", trim(lower(col("morbidades_relatadas")))) \
    .withColumn("_ingestao_timestamp", current_timestamp()) \
    .withColumn("_origem_fonte", col("unidade_saude"))

# Sobrescrevendo a Tabela Bronze com o novo volume
df_bronze_large.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("protege_bronze.pnaum_atendimentos_bruto")

print(f"✅ Tabela Bronze atualizada com sucesso!")
print(f"📊 Total de idosos (>= 60 anos) cadastrados: {df_bronze_large.count()} registros.")

🔄 Gerando e ingerindo volume populacional da PNAUM (10.000 registros)...
✅ Tabela Bronze atualizada com sucesso!
📊 Total de idosos (>= 60 anos) cadastrados: 8747 registros.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lower, trim, sha2, concat, lit, regexp_replace, 
    when, row_number, current_timestamp
)

print("🚀 Iniciando Processamento da Camada Silver (PNAUM)...")

# 1. Criar o Schema da Camada Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS protege_silver COMMENT 'Camada Silver - Triagem Clinica PNAUM (Fase 2)'")

# 2. Leitura da Tabela Delta da Camada Bronze
df_bronze = spark.table("protege_bronze.pnaum_atendimentos_bruto")

# 3. LGPD: Pseudonimização por Hash Criptográfico (SHA-256 + Salt)
SALT_SECRET = "PROTEGE_APS_PNAUM_2026_SECURE_SALT"
df_pseudo = df_bronze.withColumn(
    "cod_paciente_hash", 
    sha2(concat(col("id_paciente").cast("string"), lit(SALT_SECRET)), 256)
)

# 4. Motor de Regras de Triagem (Critérios de Beers / STOPP-START)
REGEX_BENZO = "clonazepam|diazepam|alprazolam|lorazepam|bromazepam"
REGEX_AINES = "ibuprofeno|diclofenaco|aspirina|aas|nimesulida"

df_triagem = df_pseudo.withColumn(
    "flag_mpi_beers",
    when(col("medicamento_sanitizado").rlike(REGEX_BENZO) | col("medicamento_sanitizado").rlike(REGEX_AINES), 1).otherwise(0)
).withColumn(
    "classe_farmaco_mpi",
    when(col("medicamento_sanitizado").rlike(REGEX_BENZO), "Benzodiazepinico / Anxiolitico")
    .when(col("medicamento_sanitizado").rlike(REGEX_AINES), "AINE / Anti-inflamatorio")
    .otherwise("Outros Farmacos")
)

# 5. Cruzamento Heurístico de Risco (Condição Clínica x Medicamento Prescrito)
df_silver = df_triagem.withColumn(
    "nivel_risco",
    when(
        (col("medicamento_sanitizado").rlike(REGEX_AINES)) & 
        (col("morbidades_sanitizadas").rlike("renal|insuficiencia|ulcera")),
        lit("Critico")
    ).when(
        (col("medicamento_sanitizado").rlike(REGEX_BENZO)) & 
        (col("morbidades_sanitizadas").rlike("demencia|queda|cognitiv|tontura|alzheimer")),
        lit("Critico")
    ).when(
        col("flag_mpi_beers") == 1,
        lit("Moderado")
    ).otherwise(
        lit("Adequado")
    )
).withColumn(
    "conduta_recomendada",
    when(col("nivel_risco") == "Critico", lit("Alerta Maximo: Avaliar desprescricao ou substituicao imediata."))
    .when(col("nivel_risco") == "Moderado", lit("Aviso: Monitorar reacoes adversas e funcao organica."))
    .otherwise(lit("Acompanhamento assistencial de rotina."))
).withColumn(
    "_processamento_silver_ts", current_timestamp()
)

# 6. Salvar Tabela Delta na Camada Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("protege_silver.tb_triagem_pnaum")

print("✅ Tabela protege_silver.tb_triagem_pnaum processada e salva com sucesso!")

# 7. Exibir Distribuição de Riscos Identificados na PNAUM
display(spark.sql("""
    SELECT nivel_risco, classe_farmaco_mpi, COUNT(1) AS qtd_atendimentos 
    FROM protege_silver.tb_triagem_pnaum 
    GROUP BY nivel_risco, classe_farmaco_mpi 
    ORDER BY nivel_risco DESC, qtd_atendimentos DESC
"""))

🚀 Iniciando Processamento da Camada Silver (PNAUM)...
✅ Tabela protege_silver.tb_triagem_pnaum processada e salva com sucesso!


nivel_risco,classe_farmaco_mpi,qtd_atendimentos
Moderado,Benzodiazepinico / Anxiolitico,1935
Moderado,AINE / Anti-inflamatorio,1649
Critico,AINE / Anti-inflamatorio,932
Critico,Benzodiazepinico / Anxiolitico,651
Adequado,Outros Farmacos,3580


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_date, current_timestamp, lit, expr, 
    year, month, dayofmonth, quarter, date_format
)

print("🚀 Iniciando Processamento da Camada Gold (Star Schema PNAUM)...")

# 1. Criar o Schema da Camada Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS protege_gold COMMENT 'Camada Gold - Star Schema PNAUM (Fase 2)'")

# 2. Leitura da Tabela Delta da Camada Silver
df_silver = spark.table("protege_silver.tb_triagem_pnaum")

# 3. Construção da Dimensão Paciente (com SCD Tipo 2)
df_dim_paciente = df_silver.select(
    "cod_paciente_hash", "idade", "sexo", "morbidades_sanitizadas"
).distinct() \
.withColumn("dt_inicio", current_date()) \
.withColumn("dt_fim", lit(None).cast("date")) \
.withColumn("flag_atual", lit(True))

# Salvar Dimensão Paciente Delta
df_dim_paciente.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("protege_gold.dim_paciente")

print("✅ Tabela protege_gold.dim_paciente criada com sucesso!")

# 4. Construção da Tabela Fato Triagem (Grão: Atendimento na APS)
df_fato = df_silver.select(
    col("cod_paciente_hash"),
    col("idade"),
    col("sexo"),
    col("morbidades_sanitizadas").alias("morbidades"),
    col("medicamento_sanitizado").alias("medicamento"),
    col("classe_farmaco_mpi").alias("classe_farmaco"),
    col("flag_mpi_beers"),
    col("nivel_risco"),
    col("conduta_recomendada"),
    col("dias_uso"),
    col("_origem_fonte").alias("unidade_saude"),
    col("_processamento_silver_ts").alias("data_processamento")
).withColumn("qtd_atendimentos", lit(1))

# Salvar Tabela Fato Delta
df_fato.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("protege_gold.fato_triagem_pnaum")

print("✅ Tabela protege_gold.fato_triagem_pnaum criada com sucesso!")

# 5. Exibir Resumo Consolidado do Modelo Dimensional Gold
display(spark.sql("""
    SELECT 
        nivel_risco,
        classe_farmaco,
        SUM(qtd_atendimentos) AS total_pacientes,
        ROUND(SUM(qtd_atendimentos) * 100.0 / SUM(SUM(qtd_atendimentos)) OVER(), 2) AS pct_total
    FROM protege_gold.fato_triagem_pnaum
    GROUP BY nivel_risco, classe_farmaco
    ORDER BY total_pacientes DESC
"""))

🚀 Iniciando Processamento da Camada Gold (Star Schema PNAUM)...
✅ Tabela protege_gold.dim_paciente criada com sucesso!
✅ Tabela protege_gold.fato_triagem_pnaum criada com sucesso!


nivel_risco,classe_farmaco,total_pacientes,pct_total
Adequado,Outros Farmacos,3580,40.93
Moderado,Benzodiazepinico / Anxiolitico,1935,22.12
Moderado,AINE / Anti-inflamatorio,1649,18.85
Critico,AINE / Anti-inflamatorio,932,10.66
Critico,Benzodiazepinico / Anxiolitico,651,7.44


In [0]:
from pyspark.sql import SparkSession

print("⚡ Finalizando Sprint 4: Otimização de Performance e Governança Delta...")

# 1. Executar VACUUM com tratamento de exceção para a Community Edition
try:
    spark.sql("VACUUM protege_gold.fato_triagem_pnaum RETAIN 168 HOURS")
    print("✅ VACUUM concluído com sucesso!")
except Exception as e:
    print("ℹ️ Nota de Governança: VACUUM ignorado devido às restrições do ambiente Databricks Community.")

# 2. Exibir Detalhes de Auditoria e Estatísticas da Tabela Fato Gold
display(spark.sql("""
    DESCRIBE DETAIL protege_gold.fato_triagem_pnaum
"""))

⚡ Finalizando Sprint 4: Otimização de Performance e Governança Delta...
✅ VACUUM concluído com sucesso!


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,0e41d361-ea51-4d4a-a836-2756a5c79aa2,workspace.protege_gold.fato_triagem_pnaum,null,,2026-10-14T12:23:45.398Z,2026-10-14T13:28:23.000Z,List(),List(),1,320379,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> c5e40858-cca9-4b4d-9beb-0ebe93ac09c7)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
from pyspark.sql import SparkSession

print("📊 Executando Consultas Executivas para Visualização e BI (P.R.O.T.E.G.E. Gold)...")

# 1. Visão Geral de Indicadores de Risco Populacional
print("\n--- 1. INDICADORES POPULACIONAIS DE RISCO CLÍNICO ---")
display(spark.sql("""
    SELECT 
        nivel_risco AS `Nível de Risco`,
        COUNT(1) AS `Total de Idosos`,
        ROUND(COUNT(1) * 100.0 / SUM(COUNT(1)) OVER(), 2) AS `% do Total`
    FROM protege_gold.fato_triagem_pnaum
    GROUP BY nivel_risco
    ORDER BY `% do Total` DESC
"""))

# 2. Distribuição de Riscos Críticos por Unidade de Saúde (APS)
print("\n--- 2. CRITICIDADE POR UNIDADE ATENDIMENTO (UBS) ---")
display(spark.sql("""
    SELECT 
        unidade_saude AS `Unidade de Saúde`,
        SUM(CASE WHEN nivel_risco = 'Critico' THEN 1 ELSE 0 END) AS `Atendimentos Críticos`,
        SUM(CASE WHEN nivel_risco = 'Moderado' THEN 1 ELSE 0 END) AS `Atendimentos Moderados`,
        COUNT(1) AS `Total Atendimentos`
    FROM protege_gold.fato_triagem_pnaum
    GROUP BY unidade_saude
    ORDER BY `Atendimentos Críticos` DESC
"""))

📊 Executando Consultas Executivas para Visualização e BI (P.R.O.T.E.G.E. Gold)...

--- 1. INDICADORES POPULACIONAIS DE RISCO CLÍNICO ---


Nível de Risco,Total de Idosos,% do Total
Moderado,3584,40.97
Adequado,3580,40.93
Critico,1583,18.10



--- 2. CRITICIDADE POR UNIDADE ATENDIMENTO (UBS) ---


Unidade de Saúde,Atendimentos Críticos,Atendimentos Moderados,Total Atendimentos
UBS Centro,335,744,1819
UBS Vila Real,334,691,1738
UBS Sul,316,695,1718
UBS Leste,301,749,1735
UBS Bairro Norte,297,705,1737
